# Drawing a recovered structure

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/viz.ipynb)

`andrey.viz.draw` renders a recovered graph to a self-contained SVG. The grammar is simple:
marks are drawn by shape (arrowhead, hollow circle, tail), so
color stays a redundant channel; resolved edges are a calm graphite backbone and color is reserved
for what remains uncertain.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.1.0` from PyPI;
local kernels use the installed environment.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[numba,viz]==0.1.0" pandas

In [ ]:
import numpy as np

import andrey
import andrey.viz

Recover a CPDAG from linear-Gaussian data over a four-node structure: a fork `x0 -> x1`,
`x0 -> x2`, closing into a collider at `x3`.

In [ ]:
rng = np.random.default_rng(0)
n = 2000
x0 = rng.standard_normal(n)
x1 = 0.9 * x0 + rng.standard_normal(n)
x2 = 0.8 * x0 + rng.standard_normal(n)
x3 = 0.9 * x1 + 0.9 * x2 + rng.standard_normal(n)
data = np.column_stack([x0, x1, x2, x3])

out = andrey.ges(data)
print("kind:", out.structure.kind)
print("edges:", out.structure.to_edges())

Draw it in both themes at once. `draw()` returns an `SVG`, a `str` subclass carrying a
`_repr_html_`, so composing a figure is plain string assembly: the `Panels` helper below
stitches the light and dark renders into one captioned row, and every piece it stitches is markup
`draw()` already produced. Both palettes are first-class, so a figure is never shipped in one
theme only. A bare array carries no variable names, so nodes are labeled by column index: the
collider at node `3` is oriented (both edges the graphite backbone), and whatever the CPDAG
leaves unoriented is the undirected amber.

In [ ]:
class Panels:
    """A labeled row of renders; ``_repr_html_`` lets the notebooks embed it."""

    def __init__(self, **renders: str) -> None:
        self.renders = renders

    def _repr_html_(self) -> str:
        cells = "".join(
            f"<figure>{svg}<figcaption>{label}</figcaption></figure>"
            for label, svg in self.renders.items()
        )
        return f'<div class="andrey-figure-pair">{cells}</div>'


Panels(Light=andrey.viz.draw(out.structure), Dark=andrey.viz.draw(out.structure, dark=True))

Each render is also just a string. `background=False` drops the paper fill for a transparent SVG
that composites onto an already-themed page, and the markup writes straight to a file.

In [ ]:
svg = andrey.viz.draw(out.structure, background=False)
print("markup:", svg[:60], "...")
print("bytes:", len(svg))